<a href="https://colab.research.google.com/github/gauravd12345/miniCLIP/blob/main/miniCLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [98]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("eeshawn/flickr30k")

print("Path to dataset files:", path)

images = path + '/flickr30k_images'
captions = path + '/captions.txt'

Using Colab cache for faster access to the 'flickr30k' dataset.
Path to dataset files: /kaggle/input/flickr30k


In [99]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from torchvision import transforms
from PIL import Image
from IPython.display import display

In [104]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({"eos_token": "<eos>"})

Exception ignored in: <function _ConnectionBase.__del__ at 0x7af355eff100>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 133, in __del__
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

1

In [110]:
""" Vision Transformer parameters """
H = 128               # image height
W = 128               # image width
C = 3                 # number of channels

P = 32                # patch resolution
N_p = (H * W) // P**2   # number of patches

batch_size = 64

""" Transformer parameters """
d_model = 512
d_k = 64
d_v = 64
h = 8
N = 6

vocab_len = len(tokenizer)
chunk_size = 15

In [111]:
import pandas as pd

df = pd.read_csv(captions)

image_names = df['image_name']
comment_numbers = df['comment_number']
comments = df['comment']

In [112]:
class ViTDataset(Dataset):
    def __init__(self, image_names, captions, transform=None):
        self.captions = captions
        self.image_names = image_names
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img = Image.open(f"{images}/{self.image_names[idx]}")
        if self.transform:
            img = self.transform(img)         # extract H, W, C values
        img = img.reshape(N_p, P**2 * C)        # flatten

        return img, self.captions[idx]


transform = transforms.Compose([ # resizing images
    transforms.Resize((H, W)),
    transforms.ToTensor()
])

dataset = ViTDataset(image_names, comments, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

for img, caption in dataloader:
    print(f"Samples per batch: {len(dataloader)}")
    print(f"Image batch shape: {img.shape}")
    print(f"Number of captions: {len(caption)}")
    break

Samples per batch: 2484
Image batch shape: torch.Size([64, 16, 3072])
Number of captions: 64


In [113]:
class TextEncoder(nn.Module):
  def __init__(self):
    super().__init__()

    self.embed = nn.Embedding(vocab_len, d_model)
    self.pos = nn.Embedding(chunk_size, d_model) # positional embedding

    self.W_q = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) for _ in range(N)]) # q, k, v projections
    self.W_k = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) for _ in range(N)])
    self.W_v = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_v) for _ in range(h)]) for _ in range(N)])

    self.W_o = nn.ModuleList([nn.Linear(h * d_v, d_model) for _ in range(N)]) # final projection

    self.ffn1 = nn.ModuleList([nn.Linear(d_model, 4 * d_model) for _ in range(N)]) # ffn layer
    self.ffn2 = nn.ModuleList([nn.Linear(4 * d_model, d_model) for _ in range(N)])

    self.fc = nn.Linear(d_model, vocab_len) # final layer

    self.ln1 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(N)])
    self.ln2 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(N)])

    self.dropout = nn.Dropout(0.1)

  def multi_head_attention(self, x, layer_idx):
    W_tot = []
    for Q, K, V in zip(self.W_q[layer_idx], self.W_k[layer_idx], self.W_v[layer_idx]):
      Q_i = Q(x)
      K_i = K(x)
      V_i = V(x)

      alignment = torch.matmul(Q_i, K_i.transpose(-2, -1))                      # query-key alignment

      wei = self.dropout(torch.softmax(alignment / (d_k ** 0.5), dim=-1))       # alignment weights
      wei_value = torch.matmul(wei, V_i)                                        # weighted values

      W_tot.append(wei_value)

    out = self.W_o[layer_idx](torch.cat(W_tot, dim=2))
    return out

  def forward(self, x): # (batch_size, chunk_size)
    p = torch.arange(x.size(1)).to(x.device)
    x = self.dropout(self.embed(x) + self.pos(p))     # word & positional embedding

    for i in range(N):
      out = self.multi_head_attention(x, i)  # multi head attention
      out = self.ln1[i](out + x)             # layernorm + residual connection

      fn = self.ffn1[i](out)                 # ffn
      fn = torch.relu(fn)
      fn = self.dropout(self.ffn2[i](fn))

      out = self.ln2[i](fn + out)
      x = out

    out = self.fc(out)
    return out


In [114]:
class VisionEncoder(nn.Module):
  def __init__(self):
    super().__init__()

    self.embed = nn.Embedding(vocab_len, d_model)
    self.pos = nn.Embedding(chunk_size, d_model) # positional embedding

    self.W_q = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) for _ in range(N)]) # q, k, v projections
    self.W_k = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) for _ in range(N)])
    self.W_v = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_v) for _ in range(h)]) for _ in range(N)])

    self.W_o = nn.ModuleList([nn.Linear(h * d_v, d_model) for _ in range(N)]) # final projection

    self.ffn1 = nn.ModuleList([nn.Linear(d_model, 4 * d_model) for _ in range(N)]) # ffn layer
    self.ffn2 = nn.ModuleList([nn.Linear(4 * d_model, d_model) for _ in range(N)])

    self.fc = nn.Linear(d_model, vocab_len) # final layer

    self.ln1 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(N)])
    self.ln2 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(N)])

    self.dropout = nn.Dropout(0.1)

  def multi_head_attention(self, x, layer_idx):
    W_tot = []
    for Q, K, V in zip(self.W_q[layer_idx], self.W_k[layer_idx], self.W_v[layer_idx]):
      Q_i = Q(x)
      K_i = K(x)
      V_i = V(x)

      alignment = torch.matmul(Q_i, K_i.transpose(-2, -1))                      # query-key alignment

      wei = self.dropout(torch.softmax(alignment / (d_k ** 0.5), dim=-1))       # alignment weights
      wei_value = torch.matmul(wei, V_i)                                        # weighted values

      W_tot.append(wei_value)

    out = self.W_o[layer_idx](torch.cat(W_tot, dim=2))
    return out

  def forward(self, x): # (batch_size, chunk_size)
    p = torch.arange(x.size(1)).to(x.device)
    x = self.dropout(self.embed(x) + self.pos(p))     # word & positional embedding

    for i in range(N):
      out = self.multi_head_attention(x, i)  # multi head attention
      out = self.ln1[i](out + x)             # layernorm + residual connection

      fn = self.ffn1[i](out)                 # ffn
      fn = torch.relu(fn)
      fn = self.dropout(self.ffn2[i](fn))

      out = self.ln2[i](fn + out)
      x = out

    out = self.fc(out)
    return out


In [115]:
class miniCLIP(nn.Module):
  def __init__(self):
    super().__init__()

    self.text_encoder = TextEncoder()
    self.vision_encoder = VisionEncoder

  def forward(self, x):
    text_enc = self.text_encoder(x)
    vision_enc = self.vision_encoder(x)

    return torch.matmul(text_enc, vision_enc)
